In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("amanullahasraf/covid19-pneumonia-normal-chest-xray-pa-dataset")

print("Path to dataset files:", path)

/Users/kpvarma/PycharmProjects/CNN_ViT_Hybrid_Vit_Research_Pneumonia_4_Classification/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /Users/kpvarma/.cache/kagglehub/datasets/amanullahasraf/covid19-pneumonia-normal-chest-xray-pa-dataset/versions/1


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("artyomkolas/3-kinds-of-pneumonia")

print("Path to dataset files:", path)

 80%|███████▉  | 2.79G/3.49G [17:53<04:29, 2.79MB/s] 


KeyboardInterrupt: 

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohamedasak/chest-x-ray-6-classes-dataset")

print("Path to dataset files:", path)

100%|██████████| 194M/194M [00:57<00:00, 3.52MB/s] 

Extracting files...


Path to dataset files: /Users/kpvarma/.cache/kagglehub/datasets/mohamedasak/chest-x-ray-6-classes-dataset/versions/1


In [ ]:
#Covid-2717
#Normal-2971
#Bacterial-2700
#Viral-2713

In [8]:
import os
import hashlib
import shutil
import random
from tqdm import tqdm

def get_image_hash(file_path):
    """Generates an MD5 hash to identify duplicate image content."""
    hasher = hashlib.md5()
    with open(file_path, 'rb') as f:
        buf = f.read()
        hasher.update(buf)
    return hasher.hexdigest()

def process_and_balance_data(source_root, target_root, samples_per_class=2700):
    # These folder names must match your 'Raw' subfolders exactly
    classes = ['Covid-19', 'Normal', 'Pneumonia-Bacterial', 'Pneumonia-Viral']

    if not os.path.exists(source_root):
        print(f"Error: The directory '{source_root}' does not exist.")
        return

    os.makedirs(target_root, exist_ok=True)
    hashes_found = set()
    total_copied = 0

    for cls in classes:
        source_dir = os.path.join(source_root, cls)
        target_dir = os.path.join(target_root, cls)

        if not os.path.exists(source_dir):
            print(f"⚠️ Skipping {cls}: Folder not found at {source_dir}")
            continue

        os.makedirs(target_dir, exist_ok=True)
        unique_images = []

        print(f"🔍 Checking for duplicates in {cls}...")
        file_list = [f for f in os.listdir(source_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        for filename in tqdm(file_list):
            file_path = os.path.join(source_dir, filename)
            try:
                f_hash = get_image_hash(file_path)
                if f_hash not in hashes_found:
                    hashes_found.add(f_hash)
                    unique_images.append(file_path)
            except Exception as e:
                print(f"Could not process {filename}: {e}")

        # Balancing logic: Take 2700 or whatever is available if less
        num_to_copy = min(len(unique_images), samples_per_class)
        selected_images = random.sample(unique_images, num_to_copy)

        print(f"✅ Found {len(unique_images)} unique images. Copying {num_to_copy} to Unique_Raw...")

        for img_path in selected_images:
            shutil.copy(img_path, os.path.join(target_dir, os.path.basename(img_path)))
            total_copied += 1

    print(f"\n✨ Success! {total_copied} images are now in: {target_root}")

# --- UPDATED PATHS ---
# Using the absolute path you provided to bypass the directory nesting issue
RAW_PATH = "/Users/kpvarma/PycharmProjects/CNN_ViT_Hybrid_Vit_Research_Pneumonia_4_Classification/Raw"
UNIQUE_RAW_PATH = "/Users/kpvarma/PycharmProjects/CNN_ViT_Hybrid_Vit_Research_Pneumonia_4_Classification/Unique_Raw"

process_and_balance_data(RAW_PATH, UNIQUE_RAW_PATH, samples_per_class=2700)

🔍 Checking for duplicates in Covid-19...


100%|██████████| 2717/2717 [00:00<00:00, 6577.21it/s]


✅ Found 2699 unique images. Copying 2699 to Unique_Raw...
🔍 Checking for duplicates in Normal...


100%|██████████| 2971/2971 [00:00<00:00, 6291.89it/s]


✅ Found 2970 unique images. Copying 2700 to Unique_Raw...
🔍 Checking for duplicates in Pneumonia-Bacterial...


100%|██████████| 2700/2700 [00:00<00:00, 6782.22it/s]


✅ Found 2696 unique images. Copying 2696 to Unique_Raw...
🔍 Checking for duplicates in Pneumonia-Viral...


100%|██████████| 2713/2713 [00:00<00:00, 6604.74it/s]


✅ Found 2710 unique images. Copying 2700 to Unique_Raw...

✨ Success! 10795 images are now in: /Users/kpvarma/PycharmProjects/CNN_ViT_Hybrid_Vit_Research_Pneumonia_4_Classification/Unique_Raw


In [9]:
import os
import shutil
import random
from tqdm import tqdm

def create_strictly_equal_samples(source_root, target_root, target_count=2696):
    """
    Downsamples all classes to exactly target_count using only unique,
    non-augmented images to create a clean baseline.
    """
    classes = ['Covid-19', 'Normal', 'Pneumonia-Bacterial', 'Pneumonia-Viral']

    # Ensure target directory exists
    if not os.path.exists(target_root):
        os.makedirs(target_root)
        print(f"Created directory: {target_root}")

    print(f"🚀 Starting downsampling to {target_count} images per class...")

    for cls in classes:
        source_dir = os.path.join(source_root, cls)
        target_dir = os.path.join(target_root, cls)

        if not os.path.exists(source_dir):
            print(f"⚠️ Warning: {cls} folder not found. Skipping.")
            continue

        os.makedirs(target_dir, exist_ok=True)

        # Get all unique image files
        all_images = [f for f in os.listdir(source_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        if len(all_images) < target_count:
            print(f"❌ Error: {cls} only has {len(all_images)} images, which is less than {target_count}.")
            # Fallback: take all if we can't meet the count
            selected_images = all_images
        else:
            # Randomly select exactly 2696 images
            selected_images = random.sample(all_images, target_count)

        print(f"📦 Copying {len(selected_images)} images for {cls}...")

        for img_name in tqdm(selected_images, desc=f"Transferring {cls}"):
            src_path = os.path.join(source_dir, img_name)
            dst_path = os.path.join(target_dir, img_name)
            shutil.copy(src_path, dst_path)

    print(f"\n✨ Success! Your baseline dataset is ready in: {target_root}")
    print(f"Total images: {len(classes) * target_count}")

# --- PATHS ---
UNIQUE_RAW_PATH = "/Users/kpvarma/PycharmProjects/CNN_ViT_Hybrid_Vit_Research_Pneumonia_4_Classification/Unique_Raw"
EQUAL_SAMPLE_PATH = "/Users/kpvarma/PycharmProjects/CNN_ViT_Hybrid_Vit_Research_Pneumonia_4_Classification/Equal_sample"

create_strictly_equal_samples(UNIQUE_RAW_PATH, EQUAL_SAMPLE_PATH, target_count=2696)

🚀 Starting downsampling to 2696 images per class...
📦 Copying 2696 images for Covid-19...


Transferring Covid-19: 100%|██████████| 2696/2696 [00:00<00:00, 3062.93it/s]


📦 Copying 2696 images for Normal...


Transferring Normal: 100%|██████████| 2696/2696 [00:00<00:00, 3022.80it/s]


📦 Copying 2696 images for Pneumonia-Bacterial...


Transferring Pneumonia-Bacterial: 100%|██████████| 2696/2696 [00:00<00:00, 3930.75it/s]


📦 Copying 2696 images for Pneumonia-Viral...


Transferring Pneumonia-Viral: 100%|██████████| 2696/2696 [00:00<00:00, 3451.38it/s]


✨ Success! Your baseline dataset is ready in: /Users/kpvarma/PycharmProjects/CNN_ViT_Hybrid_Vit_Research_Pneumonia_4_Classification/Equal_sample
Total images: 10784


In [ ]:
#Covid-2696
#Normal-2696
#Bacterial-2696
#Viral-2696
#Total:10784
